# 🤖 Session 2: Fetching LLM Responses (Groq + LangChain v3)

**Objective:**  
Learn how to call **Groq-hosted LLMs** via LangChain, run test prompts, and explore parameters.  

**Why This Matters:**  
- Groq provides ultra-low latency responses with models like **LLaMA 3** and **Mixtral**.  
- Before diving into RAG, participants need to practice **direct LLM calls**.  


## ✅ Step 1: Install Required Libraries
We’ll install:
- **LangChain v3**  
- **LangChain Groq integration**  


In [ ]:
!pip install -q langchain==1.0.5 langchain-groq==1.0.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.2/471.2 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.3/208.3 kB 12.3 MB/s eta 0:00:00


In [7]:
!pip show langchain langchain-groq

Name: langchain
Version: 1.0.5
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-groq
Version: 1.0.0
Summary: An integration package connecting Groq and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/groq
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: groq, langchain-core
Required-by: 


## ✅ Step 2: Setup Groq API Key
We’ll fetch it securely from **Colab Secrets**.  

In Colab:  
- Go to ` Secrets → Add new secret`  
- Key = `GROQ_API_KEY`  
- Value = your Groq key from [console.groq.com](https://console.groq.com/keys)


In [8]:
from google.colab import userdata

# Load Groq API key
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

if GROQ_API_KEY:
    print("✅ Groq API key retrieved!")
else:
    print("❌ Please add GROQ_API_KEY in Colab Secrets.")


✅ Groq API key retrieved!


## ✅ Step 3: Initialize Groq LLM
We’ll use the **ChatGroq** wrapper from LangChain.


In [9]:
from langchain_groq import ChatGroq

# Initialize with LLaMA-3 (you can also use mixtral-8x7b)
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    temperature=0.7,
    max_tokens=256
)


## ✅ Step 4: Run Basic Prompts
Test Q&A and summarization with Groq.


In [10]:
# Simple Q&A
response = llm.invoke("What is Retrieval-Augmented Generation (RAG) in AI?")
print(response)
# print("Groq Response 1:\n", response.content)

# Summarization
text = """LangChain is a framework for building AI-powered applications.
It provides tools for prompt management, chaining, agents, memory,
and integrations with external APIs and vector databases."""
summary = llm.invoke(f"Summarize this in 2 sentences:\n\n{text}")
print("\nSummary:\n", summary.content)


content='**Retrieval‑Augmented Generation (RAG)** is a family of AI architectures that combine a *retrieval* component (which fetches relevant external text from a large corpus) with a *generation* component (a language model that produces the final answer).  The idea is to let the model “look up” facts before it writes, thereby improving factual accuracy, reducing hallucinations, and making the system more data‑efficient.\n\n---\n\n## 1.' additional_kwargs={'reasoning_content': 'We need to answer: "What is Retrieval-Augmented Generation (RAG) in AI?" Provide explanation: RAG is a method combining retrieval of relevant documents and generation by language models. Provide details: architecture, components, training, advantages, applications. Also mention difference from pure generation, how retrieval works, typical models like DPR, RAG, FiD. Provide use cases: question answering, chatbots, knowledge base, etc. Provide examples: RAG by Facebook AI, RAG-Token, RAG-Sequence. Provide benefi

## ✅ Step 5: Exploring Parameters
- `temperature` → Randomness of output (0 = factual, 1 = creative).  
- `max_tokens` → Maximum response length.  


In [11]:
# Low temperature (factual)
cold = ChatGroq(model="openai/gpt-oss-20b", api_key=GROQ_API_KEY, temperature=0.0)
print("Cold Response:", cold.invoke("Suggest one creative use of AI in education").content)
print("_"*50)

# High temperature (creative)
hot = ChatGroq(model="openai/gpt-oss-20b", api_key=GROQ_API_KEY, temperature=0.9)
print("\nHot Response:", hot.invoke("Suggest one creative use of AI in education").content)


Cold Response: **AI‑Generated Interactive Storytelling Lab**

**What it is**  
A classroom‑ready, AI‑driven platform that turns every student into a co‑author. Students start with a simple prompt (e.g., “a lost robot in a city of dreams”), and the AI instantly produces a branching outline—plot beats, character arcs, and key scenes. Students then choose, edit, or rewrite any part, and the AI offers real‑time feedback, suggestions for dialogue, pacing tweaks, and even visual or audio cues. The whole process is gamified: students earn “creative credits” for exploring new narrative paths, collaborating with classmates, or integrating multimedia elements.

**How it works**  
1. **Prompt & Personality** – Students type a prompt and select a genre or tone (mystery, sci‑fi, comedy).  
2. **AI‑Generated Skeleton** – The model outputs a 3‑5 scene outline with character descriptions and conflict hooks.  
3. **Interactive Editing** – Students can click any scene to expand it, swap characters, or a

## 📝 Exercise
1. Change the model compare answers.  
2. Try summarizing a longer passage of your choice.  
3. Ask Groq to **generate pros & cons** of using RAG vs plain LLMs.  


## 🎯 Summary
- Installed **LangChain v3** with Groq integration.  
- Made simple LLM calls with `ChatGroq`.  
- Learned how to adjust parameters (`temperature`, `max_tokens`).  

**Next Notebook → Document Loaders in LangChain**  
